# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [3]:
# Task — Multi-series line with highlight

# 1. Filter: all Asian countries in the dataset
asia = df[df['Region'] == 'Asia']
highlight = 'China'          # my choice of highlighted country

# 2. Numbers for the insight title (computed from data, not hard-coded)
cn = asia[asia['Country'] == highlight].sort_values('Year')
start_val, end_val = cn['CO2_Mt'].iloc[0], cn['CO2_Mt'].iloc[-1]
start_yr, end_yr = int(cn['Year'].iloc[0]), int(cn['Year'].iloc[-1])
growth = end_val / start_val

fig = go.Figure()

# 3. Context lines: all other Asian countries, grey and thin
for country in asia['Country'].unique():
    if country == highlight:
        continue
    d = asia[asia['Country'] == country].sort_values('Year')
    fig.add_trace(go.Scatter(
        x=d['Year'], y=d['CO2_Mt'], mode='lines',
        line=dict(color='#EEEEEE', width=1),
        name=country, showlegend=False,
        hovertemplate=f"{country}<br>%{{x}}: %{{y:,.0f}} Mt<extra></extra>"
    ))

# 4. Highlighted country: coloured, thicker
fig.add_trace(go.Scatter(
    x=cn['Year'], y=cn['CO2_Mt'], mode='lines',
    line=dict(color='#D62728', width=3.5),
    name=highlight, showlegend=False,
    hovertemplate=f"{highlight}<br>%{{x}}: %{{y:,.0f}} Mt<extra></extra>"
))

# 5. Direct label at the end of the highlighted line (no legend)
fig.add_annotation(
    x=end_yr, y=end_val, text=f"<b>{highlight}</b>",
    showarrow=False, xanchor='left', xshift=8,
    font=dict(color='#D62728', size=14)
)

# 6. Clean layout: insight title, no borders, light gridlines, horizontal text
fig.update_layout(
    title=dict(
        text=f"<b>{highlight}'s CO2 emissions grew {growth:.1f}x since {start_yr}, "
             f"dwarfing every other Asian country</b>",
        x=0, xanchor='left', font=dict(size=20)
    ),
    template='plotly_white',
    plot_bgcolor='white',
    showlegend=False,
    height=520, width=900,
    margin=dict(l=70, r=100, t=90, b=60),
    xaxis=dict(title='', showgrid=False, showline=False, zeroline=False,
               tickangle=0, dtick=2),
    yaxis=dict(title='CO2 emissions (Mt)', showgrid=True, gridcolor='#F2F2F2',
               showline=False, zeroline=False, tickformat=',')
)
fig.show()
